In [5]:
import pandas as pd
import numpy as np
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

print("Imports OK")

Imports OK


In [6]:
DATA_PATH = "triage_level.csv"

df = pd.read_csv(DATA_PATH)
print(f"Raw shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Drop rows missing HPI or triage
df = df.dropna(subset=["HPI", "triage"])
print(f"\nAfter dropping nulls: {df.shape}")

Raw shape: (9149, 5)
Columns: ['stay_id', 'HPI', 'patient_info', 'initial_vitals', 'triage']

After dropping nulls: (9148, 5)


In [7]:
print("Triage distribution:")
print(df["triage"].value_counts().sort_index())

print("\nPercentages:")
print((df["triage"].value_counts(normalize=True).sort_index() * 100).round(2))

Triage distribution:
triage
1     878
2    3299
3    4907
4      64
Name: count, dtype: int64

Percentages:
triage
1     9.60
2    36.06
3    53.64
4     0.70
Name: proportion, dtype: float64


In [8]:
for i, hpi in enumerate(df["HPI"].head(3)):
    print(f"--- Example {i+1} ---")
    print(hpi[:300])
    print()

--- Example 1 ---
Patient is an ___ year-old patient with history of Sjogren's
syndrome, moderate MR, recent hospitalization for sepsis
secondary to c. diff colitis complicated by hypercarbic
respiratory failure requiring intubation, who is presenting from
___ with worsening dyspnea for one day.  Patient was
discharg

--- Example 2 ---
___ year old male with history to intestinal sarcoidosis s/p
Ex.laparoscopy, right colectomy ___ (___), ITP s/p open
splenectomy, and inguinal hernia repair complaining of new
abdominal pain that started last night and has now progressed to
___ with 1 episode of foot particulate emesis. He denies fe

--- Example 3 ---
___ w Rt AK pop to ___ bypass with NRGSV for a thrombosed
popliteal aneurysm in ___ present with worsening new onset
right foot claudication.



In [9]:
X = df["HPI"].tolist()
y = df["triage"].to_numpy()

print(f"X: {len(X)} samples")
print(f"y: {len(y)} labels")
print(f"Classes in y: {sorted(set(y))}")

X: 9148 samples
y: 9148 labels
Classes in y: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


In [10]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}")
print(f"Val:   {len(X_val)}")
print(f"Test:  {len(X_test)}")

# Verify stratification
print("\nTrain class distribution:")
print(pd.Series(y_train).value_counts().sort_index())

Train: 6403
Val:   1372
Test:  1373

Train class distribution:
1     614
2    2309
3    3435
4      45
Name: count, dtype: int64


In [11]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        stop_words="english",
        token_pattern=r"\b[a-zA-Z]{2,}\b",  # ignore ___ and numbers
        min_df=2,
        max_df=0.95,
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",   # critical for rare ESI 4
        random_state=42,
    )),
])

print("Pipeline built")

Pipeline built


In [12]:
print("Training baseline model...")
pipeline.fit(X_train, y_train)
print("Done!")

Training baseline model...
Done!


In [13]:
val_preds = pipeline.predict(X_val)

val_acc = accuracy_score(y_val, val_preds)
val_f1_macro = f1_score(y_val, val_preds, average="macro")
val_f1_weighted = f1_score(y_val, val_preds, average="weighted")

print(f"Validation Accuracy:      {val_acc:.4f}")
print(f"Validation Macro F1:      {val_f1_macro:.4f}")
print(f"Validation Weighted F1:   {val_f1_weighted:.4f}")

Validation Accuracy:      0.6086
Validation Macro F1:      0.4067
Validation Weighted F1:   0.6180


In [14]:
test_preds = pipeline.predict(X_test)

test_acc = accuracy_score(y_test, test_preds)
test_f1_macro = f1_score(y_test, test_preds, average="macro")
test_f1_weighted = f1_score(y_test, test_preds, average="weighted")

print(f"Test Accuracy:      {test_acc:.4f}")
print(f"Test Macro F1:      {test_f1_macro:.4f}")
print(f"Test Weighted F1:   {test_f1_weighted:.4f}")

Test Accuracy:      0.6089
Test Macro F1:      0.4028
Test Weighted F1:   0.6188


In [15]:
print("Classification Report (Test):")
print(classification_report(
    y_test,
    test_preds,
    labels=[1, 2, 3, 4],
    target_names=[
        "ESI 1 (Resuscitation)",
        "ESI 2 (Emergent)",
        "ESI 3 (Urgent)",
        "ESI 4 (Less Urgent)",
    ],
    zero_division=0,
))

Classification Report (Test):
                       precision    recall  f1-score   support

ESI 1 (Resuscitation)       0.29      0.42      0.34       132
     ESI 2 (Emergent)       0.54      0.54      0.54       495
       ESI 3 (Urgent)       0.77      0.70      0.73       736
  ESI 4 (Less Urgent)       0.00      0.00      0.00        10

             accuracy                           0.61      1373
            macro avg       0.40      0.41      0.40      1373
         weighted avg       0.63      0.61      0.62      1373



In [16]:
cm = confusion_matrix(y_test, test_preds, labels=[1, 2, 3, 4])

print("Confusion Matrix (rows=actual, cols=predicted):")
print("Labels order: [ESI 1, ESI 2, ESI 3, ESI 4]\n")
print(cm)

# Nicer display with pandas
cm_df = pd.DataFrame(
    cm,
    index=["Actual ESI 1", "Actual ESI 2", "Actual ESI 3", "Actual ESI 4"],
    columns=["Pred ESI 1", "Pred ESI 2", "Pred ESI 3", "Pred ESI 4"],
)
print("\n", cm_df)

Confusion Matrix (rows=actual, cols=predicted):
Labels order: [ESI 1, ESI 2, ESI 3, ESI 4]

[[ 55  60  17   0]
 [ 89 269 133   4]
 [ 46 167 512  11]
 [  1   2   7   0]]

               Pred ESI 1  Pred ESI 2  Pred ESI 3  Pred ESI 4
Actual ESI 1          55          60          17           0
Actual ESI 2          89         269         133           4
Actual ESI 3          46         167         512          11
Actual ESI 4           1           2           7           0


In [17]:
def predict_sample(text):
    pred = int(pipeline.predict([text])[0])
    proba = pipeline.predict_proba([text])[0]
    confidence = float(max(proba))
    return pred, confidence

samples = [
    "Severe chest pain and difficulty breathing",
    "Lethargy and confusion for two days",
    "Abdominal pain with vomiting",
    "Mild rash on left arm",
]

for s in samples:
    pred, conf = predict_sample(s)
    print(f"Text: {s}")
    print(f"  → ESI {pred} (confidence: {conf:.2%})")
    print()

Text: Severe chest pain and difficulty breathing
  → ESI 2 (confidence: 54.61%)

Text: Lethargy and confusion for two days
  → ESI 2 (confidence: 50.57%)

Text: Abdominal pain with vomiting
  → ESI 3 (confidence: 75.10%)

Text: Mild rash on left arm
  → ESI 1 (confidence: 46.22%)



In [18]:
import os
os.makedirs("../models", exist_ok=True)

joblib.dump(pipeline, "../models/baseline_model.pkl")
print("Model saved to ../models/baseline_model.pkl")

# Verify
loaded = joblib.load("../models/baseline_model.pkl")
test_pred = loaded.predict(["Severe chest pain"])[0]
print(f"Reload test prediction: ESI {test_pred}")

Model saved to ../models/baseline_model.pkl
Reload test prediction: ESI 2


In [19]:
print("=" * 60)
print("BASELINE MODEL SUMMARY")
print("=" * 60)
print(f"Input feature:  HPI text only")
print(f"Target:         ESI triage level (1-4)")
print(f"Training size:  {len(X_train)}")
print(f"Val size:       {len(X_val)}")
print(f"Test size:      {len(X_test)}")
print(f"Model:          TF-IDF + Logistic Regression")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Macro F1:  {test_f1_macro:.4f}")
print("=" * 60)

BASELINE MODEL SUMMARY
Input feature:  HPI text only
Target:         ESI triage level (1-4)
Training size:  6403
Val size:       1372
Test size:      1373
Model:          TF-IDF + Logistic Regression
Test Accuracy:  0.6089
Test Macro F1:  0.4028


In [20]:
from google.colab import files
files.download("../models/baseline_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>